# datalab-plot starter

Pick a set of cells from a datalab instance, compare their cycling performance, and drill into individual cells.

## 0. Configure the connection

Two things are needed:

1. **`DATALAB_URL`** — the base URL of your datalab instance.
2. **`DATALAB_API_KEY`** — a personal API token. Generate one by logging into the instance in a browser and visiting its `/get-api-key` endpoint (e.g. `https://datalab.example.org/get-api-key`), or the **Account** page if your instance exposes it.

Both are read from environment variables. The cell below defaults `DATALAB_URL` and prompts for the API key **only** if it isn't already exported in your shell — so colleagues can either `export DATALAB_API_KEY=...` before launching Jupyter or paste it once per kernel session. The token stays in process memory; this notebook does not write it to disk.

In [ ]:
import os
import getpass

# Change this to your instance, or `export DATALAB_URL=...` in your shell before launching Jupyter.
os.environ.setdefault("DATALAB_URL", "https://datalab.example.org/")

if not os.environ.get("DATALAB_API_KEY"):
    os.environ["DATALAB_API_KEY"] = getpass.getpass(
        f"Enter your API key for {os.environ['DATALAB_URL']} (hidden): "
    )

print("Connected to:", os.environ["DATALAB_URL"])

In [ ]:
from datalab_plot import plot_cycles, plot_cell, find_cells
from datalab_plot.picker import pick_cells

# Neware .nda/.ndax INFO logs are silenced automatically when datalab_plot is imported.

## 1. Find items

`find_cells` returns a DataFrame — use it to scan what's on the instance. Filter with a free-text `query` or leave it blank to list everything.

In [ ]:
find_cells(query="cel", limit=30)

## 2. Pick cells interactively

The widget below lists matching items. **Highlighted rows = the current selection.** They are exactly what `picker.selected` returns, and exactly what the next `plot_cycles(picker.selected)` call will plot.

- Click a row to select it.
- `Cmd` / `Ctrl`-click to toggle additional rows.
- `Shift`-click to select a contiguous range.
- A live counter under the list shows the chosen item_ids; the picker does **not** re-plot automatically — re-run the plot cells below after changing the selection.

The widget inherits the JupyterLab theme — dark mode renders dark, light mode renders light.

In [ ]:
picker = pick_cells(query="cel", limit=300)

## 3. Comparison plots

Four modes:
- `"summary"` — discharge capacity and Coulombic efficiency vs cycle. Shared legend below both panels, so it stays out of the data even with many cells.
- `"voltage_capacity"` — voltage curves at a chosen cycle.
- `"dqdv"` — differential capacity at a chosen cycle.
- `"voltage_time"` — voltage vs elapsed time, one trace per cell.

Each call returns a `matplotlib.figure.Figure` — assign it to a variable and call `.savefig(...)` to export, or `plt.show()` to display.

In [ ]:
plot_cycles(picker.selected, mode="summary");

In [ ]:
plot_cycles(picker.selected, mode="voltage_capacity", cycle=1);

In [ ]:
plot_cycles(picker.selected, mode="voltage_time");

## 4. Single-cell deep dive

Pick one item_id from the picker and look at its full cycling. `plot_cell` colour-codes traces by cycle index using the viridis colormap, so capacity fade across the cycle stack is visible at a glance.

Modes: `voltage_time`, `voltage_capacity`, `dqdv`, `summary`.

In [ ]:
item_id = next(iter(picker.selected.values()))
plot_cell(item_id, mode="voltage_capacity");